In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import sys; sys.path.append("..")
from src.data.harmonise import prepare_fitzpatrick
from pathlib import Path

In [5]:
RAW = Path("..") / "data" / "raw"
for folder in sorted(RAW.iterdir()):
    print(folder.name, "→", len(list(folder.rglob("*"))), "files")

.DS_Store → 0 files
ddi → 657 files
fitzpatrick17k → 16580 files
isic → 25333 files


In [6]:
df = prepare_fitzpatrick(RAW / "fitzpatrick17k" / "fitzpatrick17ku.csv",
                                 RAW / "fitzpatrick17k")
df.shape

raw rows:                                   16577
after dropping unknown skin:                16012
after keeping benign/malignant skin:        4320


(4320, 5)

In [7]:
df.head()

,image_path,target,skin_group,group_id,source
0,../data/raw/fitzpatrick17k/d2bac3c9e4499032ca8...,0,light,d2bac3c9e4499032ca8e9b07c7d3bc40,fitzpatrick17k
1,../data/raw/fitzpatrick17k/45f7fe0e10214e32e89...,1,dark,45f7fe0e10214e32e890cad9d29d4811,fitzpatrick17k
2,../data/raw/fitzpatrick17k/b87804452f60aa162a6...,1,light,b87804452f60aa162a6d29c0f66a2466,fitzpatrick17k
3,../data/raw/fitzpatrick17k/4c3f795cf8eb72b946f...,1,dark,4c3f795cf8eb72b946f9bd2642cf23c1,fitzpatrick17k
4,../data/raw/fitzpatrick17k/99247c9fe486aa9ab71...,0,light,99247c9fe486aa9ab71686c8e676c135,fitzpatrick17k


In [8]:
strat_key = df["skin_group"] + "_" + df["target"].astype(str)
print(strat_key)

0        light_0
1         dark_1
2        light_1
3         dark_1
4        light_0
          ...   
4315    medium_1
4316    medium_1
4317    medium_0
4318    medium_1
4319    medium_0
Length: 4320, dtype: str


In [9]:
temp_df, test_df = train_test_split(df, test_size = 0.15, random_state  = 42, stratify = strat_key)
print(temp_df.shape)
print(test_df.shape)

(3672, 5)
(648, 5)


In [10]:
strat_key_temp = temp_df["skin_group"] + "_" + temp_df["target"].astype(str)
print(strat_key_temp)

3721      dark_1
471      light_1
4193    medium_1
600      light_1
1321     light_1
          ...   
2333     light_1
230     medium_1
2654     light_1
2651    medium_1
513       dark_0
Length: 3672, dtype: str


In [11]:
train_df, val_df = train_test_split(temp_df, test_size = 0.15, random_state  = 42, stratify = strat_key_temp)
print(train_df.shape)
print(val_df.shape)

(3121, 5)
(551, 5)


In [12]:
train_df = train_df.assign(split = "train")
val_df   = val_df.assign(split = "val")
test_df  = test_df.assign(split = "test")

print(train_df.shape)

(3121, 6)


In [13]:
new_df = (
    pd.concat([train_df, val_df, test_df])
      .sort_index()          # back to original row order
      .reset_index(drop=True)
)

In [14]:
new_df["split"].value_counts()

split
train    3121
test      648
val       551
Name: count, dtype: int64

In [15]:
pd.crosstab([new_df["skin_group"], new_df["target"]], new_df["split"])

split              test  train  val
skin_group target                  
dark       0         31    146   26
           1         31    150   27
light      0        167    806  142
           1        179    864  152
medium     0        126    609  107
           1        114    546   97

In [16]:
train_ids, val_ids, test_ids = (set(df["group_id"]) for df in (train_df, val_df, test_df))

overlaps = {
    "train∩val":  train_ids & val_ids,
    "train∩test": train_ids & test_ids,
    "val∩test":   val_ids & test_ids,
}

assert all(len(v) == 0 for v in overlaps.values()), \
    "Leakage detected: " + ", ".join(f"{k}={sorted(v)[:10]}" for k, v in overlaps.items() if v)

print("✓ No group_id leakage across splits")

✓ No group_id leakage across splits


In [17]:
new_df[["group_id", "split"]].sort_values(["split", "group_id"]).to_csv(
    "../splits/fitzpatrick17k_splits.csv", index = False
)

In [18]:
from src.data.harmonise import prepare_fitzpatrick
from src.data.splits import make_splits, save_splits
from src.config import RAW, RANDOM_SEED, VAL_FRAC, TEST_FRAC, SPLITS_DIR

In [19]:
csv_path = RAW / "fitzpatrick17k" / "fitzpatrick17ku.csv"
img_dir = RAW / "fitzpatrick17k"
fitz_clean = prepare_fitzpatrick(csv_path, img_dir)
print(fitz_clean.shape)

raw rows:                                   16577
after dropping unknown skin:                16012
after keeping benign/malignant skin:        4320
(4320, 5)


In [20]:
fitz_split = make_splits(fitz_clean, RANDOM_SEED, VAL_FRAC, TEST_FRAC)

In [21]:
print(fitz_split["split"].value_counts())

split
train    3024
val       648
test      648
Name: count, dtype: int64


In [22]:
save_splits(fitz_split, SPLITS_DIR / "fitzpatrick17k_splits.csv")

In [23]:
ct = pd.crosstab([fitz_split["skin_group"], fitz_split["target"]], fitz_split["split"])
ct.reindex(["light", "medium", "dark"], level=0)[["train", "val", "test"]]

split              train  val  test
skin_group target                  
light      0         781  167   167
           1         837  179   179
medium     0         589  127   126
           1         529  114   114
dark       0         142   30    31
           1         146   31    31